In [11]:
import os
import csv
import pandas as pd
import re
import subprocess
import numpy as np
import scipy.stats as stats
import scipy.sparse as sp

new_dir = "/home/jingqi/RNALocateV3.0/Data"
os.chdir(new_dir)
os.getcwd()

'/home/jingqi/RNALocateV3.0/Data'

### Overlapping

In [4]:
file_attract = "ATTRACT/FIMO_Processed/Functional_Zipcode_Candidates.csv"
file_meme = "FIMO_New/Aginst_MEME_file/Functional_Zipcode_Candidates.csv" 

out_overlap = "ATTRACT/FIMO_Processed/FZC_Overlapped.csv"

print("Loading candidate datasets...")
try:
    df_attract = pd.read_csv(file_attract)
    df_meme = pd.read_csv(file_meme)
except FileNotFoundError as e:
    print(f"[FATAL ERROR] Could not find file: {e.filename}")
    exit()

# Extract unique genes from both datasets
genes_attract = set(df_attract['Gene'].dropna())
genes_meme = set(df_meme['Gene'].dropna())

# Calculate the intersection
overlapping_genes = genes_attract.intersection(genes_meme)

# Filter the dataframe to only include the overlapping genes 
# (Using df_attract here, but the gene list is identical)
df_overlap = df_attract[df_attract['Gene'].isin(overlapping_genes)].copy()

# Ensure the output directory exists
os.makedirs(os.path.dirname(out_overlap), exist_ok=True)

# Save the strictly overlapping dataset
df_overlap.to_csv(out_overlap, index=False)

print("\n--- COMPARISON COMPLETE ---")
print(f"Candidates in ATtRACT pipeline: {len(genes_attract)}")
print(f"Candidates in MEME pipeline:    {len(genes_meme)}")
print(f"Strict Overlap (High Confidence): {len(overlapping_genes)}")
print(f"\nFinal filtered dataset saved to: {out_overlap}")

Loading candidate datasets...

--- COMPARISON COMPLETE ---
Candidates in ATtRACT pipeline: 78
Candidates in MEME pipeline:    88
Strict Overlap (High Confidence): 64

Final filtered dataset saved to: ATTRACT/FIMO_Processed/FZC_Overlap_AgainstMEME.csv


### Adding up

In [7]:
out_union = "ATTRACT/FIMO_Processed/FZC_Combined.csv"

print("Loading candidate datasets...")
try:
    df_attract = pd.read_csv(file_attract)
    df_meme = pd.read_csv(file_meme)
except FileNotFoundError as e:
    print(f"[FATAL ERROR] Could not find file: {e.filename}")
    exit()

print("Concatenating datasets...")
df_combined = pd.concat([df_attract, df_meme], ignore_index=True)

# CORRECTED: Drop exact duplicate rows, but keep all unique transcripts per gene
df_combined_unique = df_combined.drop_duplicates(subset=['Gene', 'Transcript'], keep='first')

os.makedirs(os.path.dirname(out_union), exist_ok=True)
df_combined_unique.to_csv(out_union, index=False)

genes_attract = df_attract['Gene'].nunique()
genes_meme = df_meme['Gene'].nunique()
total_unique_genes = df_combined_unique['Gene'].nunique()
total_transcripts = len(df_combined_unique)

print("\n--- COMBINATION COMPLETE ---")
print(f"Candidates in ATtRACT pipeline: {genes_attract} genes")
print(f"Candidates in MEME pipeline:    {genes_meme} genes")
print(f"Total Unique Combined Genes:      {total_unique_genes}")
print(f"Total Unique Transcripts Retained: {total_transcripts}")
print(f"\nFinal comprehensive dataset saved to: {out_union}")

Loading candidate datasets...
Concatenating datasets...

--- COMBINATION COMPLETE ---
Candidates in ATtRACT pipeline: 78 genes
Candidates in MEME pipeline:    88 genes
Total Unique Combined Genes:      102
Total Unique Transcripts Retained: 315

Final comprehensive dataset saved to: ATTRACT/FIMO_Processed/FZC_Combined.csv


## Cluster genes in cell types

In [9]:
import scanpy as sc

adata = sc.read_h5ad("/home/jingqi/isoforms/adata_thresholded.h5ad")

### Assigning and Clustering

In [12]:
output_dir = "ATTRACT/FIMO_Processed/CellType_Lists"
df_candidates = df_candidates = pd.read_csv(out_union)
os.makedirs(output_dir, exist_ok=True)

CELL_TYPE_COL = "assignments" 
ABSOLUTE_CPM_MIN = 15
DOMINANCE_FRACTION = 0.65

# Clean transcript names to match the AnnData variables
df_candidates['Clean_Transcript'] = df_candidates['Transcript'].str.split('.').str[0]

# 1. REBUILD PSEUDO-BULK CPM MATRIX
print("Rebuilding pseudo-bulk CPM matrix...")
if sp.issparse(adata.X):
    raw_counts = adata.X.expm1()
else:
    raw_counts = np.expm1(adata.X)

unique_cell_types = adata.obs[CELL_TYPE_COL].unique()
cell_type_sums = []

for ct in unique_cell_types:
    cells_mask = (adata.obs[CELL_TYPE_COL] == ct).values
    ct_sum = raw_counts[cells_mask].sum(axis=0)
    cell_type_sums.append(np.asarray(ct_sum).flatten())

pseudo_bulk_matrix = np.vstack(cell_type_sums)
library_sizes = pseudo_bulk_matrix.sum(axis=1, keepdims=True)
library_sizes[library_sizes == 0] = 1 
cpm_matrix = (pseudo_bulk_matrix / library_sizes) * 1e6

cpm_df = pd.DataFrame(
    cpm_matrix, 
    index=unique_cell_types, 
    columns=adata.var_names.astype(str).str.split('.').str[0]
)

# 2. APPLY DOMINANCE SORTING ENGINE
print(f"Sorting genes by Cell Type (Abs > {ABSOLUTE_CPM_MIN} CPM, Dominance > {DOMINANCE_FRACTION})...")

# Dictionary to hold the final assigned genes for each cell type
cell_type_assignments = {ct: [] for ct in unique_cell_types}

for gene, group in df_candidates.groupby('Gene'):
    transcripts = group['Clean_Transcript'].values
    valid_transcripts = [t for t in transcripts if t in cpm_df.columns]
    
    if not valid_transcripts:
        continue
        
    gene_cpm_matrix = cpm_df[valid_transcripts]
    
    # Evaluate the gene across every cell type
    for cell_type in unique_cell_types:
        ct_expression = gene_cpm_matrix.loc[cell_type]
        total_gene_cpm = ct_expression.sum()
        
        if total_gene_cpm == 0:
            continue
            
        max_transcript = ct_expression.idxmax()
        max_cpm = ct_expression.max()
        fraction = max_cpm / total_gene_cpm
        
        # The Two-Part Gate
        if max_cpm >= ABSOLUTE_CPM_MIN and fraction >= DOMINANCE_FRACTION:
            # Extract the corresponding SCL prediction to include in the output
            transcript_data = group[group['Clean_Transcript'] == max_transcript].iloc[0]
            scl_profile = transcript_data.get('Adjusted_SCL_Profile', 'Unknown')
            
            cell_type_assignments[cell_type].append({
                'Gene': gene,
                'Dominant_Transcript': max_transcript,
                'Absolute_CPM': round(max_cpm, 2),
                'Dominance_Fraction': round(fraction, 3),
                'Localized_To': scl_profile
            })


print("\n--- SORTING COMPLETE ---")
total_assigned = 0

for cell_type, records in cell_type_assignments.items():
    if not records:
        continue
        
    df_ct = pd.DataFrame(records)
    df_ct = df_ct.sort_values(by='Dominance_Fraction', ascending=False)
    
    # Clean the cell type name for safe file saving
    safe_ct_name = str(cell_type).replace('/', '_').replace(' ', '_')
    out_path = os.path.join(output_dir, f"{safe_ct_name}_STRING_List.csv")
    
    df_ct.to_csv(out_path, index=False)
    total_assigned += len(records)
    print(f"{cell_type}: {len(records)} active genes written to {safe_ct_name}_STRING_List.csv")

print(f"\nSuccessfully executed sorting. {total_assigned} total cell-type specific assignments were made.")
print(f"You can find the isolated gene lists for STRING analysis in: {output_dir}/")

Rebuilding pseudo-bulk CPM matrix...
Sorting genes by Cell Type (Abs > 15 CPM, Dominance > 0.65)...

--- SORTING COMPLETE ---
none: 35 active genes written to none_STRING_List.csv
Sensory: 43 active genes written to Sensory_STRING_List.csv
SatGlia: 35 active genes written to SatGlia_STRING_List.csv
BCC: 36 active genes written to BCC_STRING_List.csv
Melanocytes: 31 active genes written to Melanocytes_STRING_List.csv
NCC: 36 active genes written to NCC_STRING_List.csv
Mesenchyme: 25 active genes written to Mesenchyme_STRING_List.csv
ChC: 37 active genes written to ChC_STRING_List.csv
Gut_neuron: 39 active genes written to Gut_neuron_STRING_List.csv
SC: 33 active genes written to SC_STRING_List.csv
tSC: 28 active genes written to tSC_STRING_List.csv
Gut_glia: 36 active genes written to Gut_glia_STRING_List.csv
Symp: 37 active genes written to Symp_STRING_List.csv
nmSC: 31 active genes written to nmSC_STRING_List.csv
mSC: 40 active genes written to mSC_STRING_List.csv
enFib: 35 active gen

In [13]:
import glob

input_dir = "ATTRACT/FIMO_Processed/CellType_Lists"
output_unified_csv = "ATTRACT/FIMO_Processed/CellType_Lists/Unified_CellType_Genes.csv"

file_pattern = os.path.join(input_dir, "*_STRING_List.csv")
csv_files = glob.glob(file_pattern)

cell_type_genes = {}

for file in csv_files:
    filename = os.path.basename(file)
    cell_type = filename.replace("_STRING_List.csv", "")
    
    df = pd.read_csv(file)
    if 'Gene' in df.columns:
        cell_type_genes[cell_type] = df['Gene'].tolist()

if not cell_type_genes:
    print(f"Error: No valid files found in {input_dir}.")
else:
    unified_df = pd.concat([pd.Series(genes, name=ct) for ct, genes in cell_type_genes.items()], axis=1)
    unified_df.to_csv(output_unified_csv, index=False)
    print(f"Unified file successfully generated with {len(unified_df.columns)} cell types.")
    print(f"Saved to: {output_unified_csv}")

Unified file successfully generated with 16 cell types.
Saved to: ATTRACT/FIMO_Processed/CellType_Lists/Unified_CellType_Genes.csv


## Cluster genes based on RBP

In [16]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from collections import defaultdict

# --- CONFIGURATION ---
input_csv = "ATTRACT/FIMO_Processed/Functional_Zipcode_Candidates.csv"
output_csv = "ATTRACT/FIMO_Processed/RBP_Lists/RBP_Driven_Clusters.csv"

df = pd.read_csv(input_csv)

# Global tracking structures
rbp_gene_assignments = defaultdict(set)
rbp_scl_scoreboard = defaultdict(lambda: defaultdict(int))

def get_motifs(motif_str):
    if pd.isna(motif_str) or motif_str == 'None': return set()
    return set(x.strip() for x in str(motif_str).split(',') if x.strip())

def get_scls(scl_str):
    if pd.isna(scl_str) or scl_str == 'Unclassified': return set()
    return set(x.strip() for x in str(scl_str).split(',') if x.strip())

print("Executing RBP Pattern Clustering and Global SCL Voting...")

for gene, group in df.groupby('Gene'):
    transcripts = group.reset_index(drop=True)
    n_transcripts = len(transcripts)
    
    if n_transcripts < 2:
        continue
        
    # Build Motif Presence/Absence Matrix for the Gene
    all_motifs = list(set.union(*[get_motifs(m) for m in transcripts['Motif_Profile']]))
    if not all_motifs:
        continue
        
    matrix = np.zeros((n_transcripts, len(all_motifs)))
    for i, row in transcripts.iterrows():
        t_motifs = get_motifs(row['Motif_Profile'])
        for j, motif in enumerate(all_motifs):
            if motif in t_motifs:
                matrix[i, j] = 1
                
    # 1. Pattern Division (K=2)
    if n_transcripts == 2:
        labels = np.array([0, 1])
    else:
        # Prevent failure if all transcripts are structurally identical
        if np.all(matrix == matrix[0]):
            continue
        clusterer = AgglomerativeClustering(n_clusters=2, metric='cosine', linkage='average')
        labels = clusterer.fit_predict(matrix)
        
        # If clustering fails to separate them, skip
        if len(set(labels)) < 2:
            continue
            
    # 2. Select Representative Transcript for Each Pattern
    reps = {}
    for cluster_id in [0, 1]:
        indices = np.where(labels == cluster_id)[0]
        if len(indices) == 1:
            reps[cluster_id] = indices[0]
        else:
            sub_matrix = matrix[indices]
            sim_matrix = cosine_similarity(sub_matrix)
            # Find the transcript with the highest average similarity to others in its pattern
            avg_sim = sim_matrix.mean(axis=1)
            best_idx_local = np.argmax(avg_sim)
            reps[cluster_id] = indices[best_idx_local]
            
    rep_A_idx, rep_B_idx = reps[0], reps[1]
    
    motifs_A = get_motifs(transcripts.iloc[rep_A_idx]['Motif_Profile'])
    motifs_B = get_motifs(transcripts.iloc[rep_B_idx]['Motif_Profile'])
    
    scls_A = get_scls(transcripts.iloc[rep_A_idx]['Adjusted_SCL_Profile'])
    scls_B = get_scls(transcripts.iloc[rep_B_idx]['Adjusted_SCL_Profile'])
    
    # 3. Subtraction across the Representatives
    unique_motifs_A = motifs_A - motifs_B
    unique_motifs_B = motifs_B - motifs_A
    unique_scls_A = scls_A - scls_B
    unique_scls_B = scls_B - scls_A
    
    # 4. Global Scoreboard Assignment
    # If Pattern A has unique SCLs, assign points to Pattern A's unique RBPs
    if unique_scls_A:
        for rbp in unique_motifs_A:
            rbp_gene_assignments[rbp].add(gene)
            for scl in unique_scls_A:
                rbp_scl_scoreboard[rbp][scl] += 1
                
    # Repeat for Pattern B
    if unique_scls_B:
        for rbp in unique_motifs_B:
            rbp_gene_assignments[rbp].add(gene)
            for scl in unique_scls_B:
                rbp_scl_scoreboard[rbp][scl] += 1

# 5. Resolve the Scoreboard and Format Output (UPDATED FOR TOP 3)
records = []
for rbp in rbp_scl_scoreboard.keys():
    scl_counts = rbp_scl_scoreboard[rbp]
    total_votes = sum(scl_counts.values())
    
    if total_votes == 0:
        continue
        
    # Sort all SCLs for this RBP by their vote count (highest to lowest)
    sorted_scls = sorted(scl_counts.items(), key=lambda item: item[1], reverse=True)
    
    # Strictly select the Top 3 (or fewer, if it only has 1 or 2)
    top_scls = sorted_scls[:3]
    
    # Format the output string (e.g., "Nucleus (42) | Cytoplasm (12) | Membrane (3)")
    formatted_scls = [f"{scl} ({votes})" for scl, votes in top_scls]
    
    records.append({
        'RBP_Motif': rbp,
        'Target_Gene_Count': len(rbp_gene_assignments[rbp]),
        'Regulated_Genes': ", ".join(sorted(list(rbp_gene_assignments[rbp]))),
        'Driven_Destinations': " | ".join(formatted_scls),
        'Total_SCL_Votes': total_votes
    })

# Export
df_out = pd.DataFrame(records)
df_out = df_out.sort_values(by='Target_Gene_Count', ascending=False)

os.makedirs(os.path.dirname(output_csv), exist_ok=True)
df_out.to_csv(output_csv, index=False)

print(f"Network isolated: {len(df_out)} unique RBPs driving differential localization.")
print(f"Results saved to: {output_csv}")

Executing RBP Pattern Clustering and Global SCL Voting...
Network isolated: 18 unique RBPs driving differential localization.
Results saved to: ATTRACT/FIMO_Processed/RBP_Lists/RBP_Driven_Clusters.csv


## Cluster on RBP + Cell type

In [ ]:
## Four threshold
# 1. within a pattern, if there is at least one transcript has high absolut expression in
# most of cell type (10/16), completely silence that pattern (housekeeping isoforms)
# 2. for the normal pattern, choose the highst absolut expression in each cell type
# 3. the highest isoform has to take up at least 40% of total gene expression
# 4. the pattern expression in that cell type has to be at least 1.5 times than in others

In [30]:
rbp_network_csv = "ATTRACT/FIMO_Processed/RBP_Lists/RBP_Driven_Clusters.csv" 
combined_candidates_csv = "ATTRACT/FIMO_Processed/Functional_Zipcode_Candidates.csv"
output_csv = "ATTRACT/FIMO_Processed/RBP_Lists/RBP-CellType_lists.csv"


ABSOLUTE_CPM_MIN = 15
RELATIVE_THRESHOLD = 0.20   
HOUSEKEEPING_MAX_CELLS = 8   
FOLD_CHANGE_MIN = 1.2       

# Helper functions
def get_motifs(motif_str):
    if pd.isna(motif_str) or motif_str == 'None': return set()
    return set(x.strip() for x in str(motif_str).split(',') if x.strip())

def get_scls(scl_str):
    if pd.isna(scl_str) or scl_str == 'Unclassified': return set()
    return set(x.strip() for x in str(scl_str).split(',') if x.strip())

# 1. Load Data & Rebuild CPM Matrix
print("Loading data and building pseudo-bulk matrices...")
df = pd.read_csv(input_csv)
df['Clean_Transcript'] = df['Transcript'].str.split('.').str[0]

gene_to_all_transcripts = df.groupby('Gene')['Clean_Transcript'].unique().to_dict()

if sp.issparse(adata.X):
    raw_counts = adata.X.expm1()
else:
    raw_counts = np.expm1(adata.X)

unique_cell_types = adata.obs['assignments'].unique()
cell_type_sums = [np.asarray(raw_counts[(adata.obs['assignments'] == ct).values].sum(axis=0)).flatten() for ct in unique_cell_types]
pseudo_bulk_matrix = np.vstack(cell_type_sums)
library_sizes = np.where(pseudo_bulk_matrix.sum(axis=1, keepdims=True) == 0, 1, pseudo_bulk_matrix.sum(axis=1, keepdims=True))
cpm_matrix = (pseudo_bulk_matrix / library_sizes) * 1e6

cpm_df = pd.DataFrame(cpm_matrix, index=unique_cell_types, columns=adata.var_names.astype(str).str.split('.').str[0])

# Global Tracking Structures
rbp_scl_scoreboard = defaultdict(lambda: defaultdict(int))
rbp_gene_driven_transcripts = defaultdict(lambda: defaultdict(list)) 

print("Executing Pattern Clustering and Subtraction...")
for gene, group in df.groupby('Gene'):
    transcripts = group.reset_index(drop=True)
    n_transcripts = len(transcripts)
    
    if n_transcripts < 2: continue
        
    all_motifs = list(set.union(*[get_motifs(m) for m in transcripts['Motif_Profile']]))
    if not all_motifs: continue
        
    matrix = np.zeros((n_transcripts, len(all_motifs)))
    for i, row in transcripts.iterrows():
        t_motifs = get_motifs(row['Motif_Profile'])
        for j, motif in enumerate(all_motifs):
            if motif in t_motifs: matrix[i, j] = 1
                
    if n_transcripts == 2:
        labels = np.array([0, 1])
    else:
        if np.all(matrix == matrix[0]): continue
        clusterer = AgglomerativeClustering(n_clusters=2, metric='cosine', linkage='average')
        labels = clusterer.fit_predict(matrix)
        if len(set(labels)) < 2: continue
            
    reps = {}
    transcripts_in_pattern = {0: [], 1: []}
    
    for cluster_id in [0, 1]:
        indices = np.where(labels == cluster_id)[0]
        transcripts_in_pattern[cluster_id] = transcripts.iloc[indices]['Clean_Transcript'].tolist()
        
        if len(indices) == 1: reps[cluster_id] = indices[0]
        else:
            sub_matrix = matrix[indices]
            sim_matrix = cosine_similarity(sub_matrix)
            reps[cluster_id] = indices[np.argmax(sim_matrix.mean(axis=1))]
            
    rep_A_idx, rep_B_idx = reps[0], reps[1]
    
    unique_motifs_A = get_motifs(transcripts.iloc[rep_A_idx]['Motif_Profile']) - get_motifs(transcripts.iloc[rep_B_idx]['Motif_Profile'])
    unique_motifs_B = get_motifs(transcripts.iloc[rep_B_idx]['Motif_Profile']) - get_motifs(transcripts.iloc[rep_A_idx]['Motif_Profile'])
    
    unique_scls_A = get_scls(transcripts.iloc[rep_A_idx]['Adjusted_SCL_Profile']) - get_scls(transcripts.iloc[rep_B_idx]['Adjusted_SCL_Profile'])
    unique_scls_B = get_scls(transcripts.iloc[rep_B_idx]['Adjusted_SCL_Profile']) - get_scls(transcripts.iloc[rep_A_idx]['Adjusted_SCL_Profile'])
    
    if unique_scls_A:
        for rbp in unique_motifs_A:
            rbp_gene_driven_transcripts[rbp][gene].extend(transcripts_in_pattern[0])
            for scl in unique_scls_A: rbp_scl_scoreboard[rbp][scl] += 1
                
    if unique_scls_B:
        for rbp in unique_motifs_B:
            rbp_gene_driven_transcripts[rbp][gene].extend(transcripts_in_pattern[1])
            for scl in unique_scls_B: rbp_scl_scoreboard[rbp][scl] += 1

rbp_destinations = {}
for rbp, scl_counts in rbp_scl_scoreboard.items():
    sorted_scls = sorted(scl_counts.items(), key=lambda item: item[1], reverse=True)[:3]
    rbp_destinations[rbp] = " | ".join([f"{scl} ({votes})" for scl, votes in sorted_scls])

# Thresholds
print("Applying Assignment Gates (Absolute, Relative, Housekeeping Silencer, Fold-Change)...")
rbp_cell_genes = defaultdict(lambda: defaultdict(set))

for rbp, gene_dict in rbp_gene_driven_transcripts.items():
    for gene, specific_transcripts in gene_dict.items():
        
        valid_pattern_transcripts = [t for t in specific_transcripts if t in cpm_df.columns]
        all_gene_transcripts = [t for t in gene_to_all_transcripts.get(gene, []) if t in cpm_df.columns]
        
        if not valid_pattern_transcripts or not all_gene_transcripts: continue
            
        pattern_cpm_matrix = cpm_df[valid_pattern_transcripts]
        total_gene_cpm_matrix = cpm_df[all_gene_transcripts]
        
        # GATE 1: The Housekeeping Silencer
        is_housekeeping = False
        for t in valid_pattern_transcripts:
            # Check how many cell types have this transcript >= ABSOLUTE_CPM_MIN
            if (pattern_cpm_matrix[t] >= ABSOLUTE_CPM_MIN).sum() > HOUSEKEEPING_MAX_CELLS:
                is_housekeeping = True
                break
                
        if is_housekeeping:
            continue  # Silence this pattern entirely
            
        # Calculate max pattern transcript for every cell type for the fold-change calculation
        pattern_max_series = pattern_cpm_matrix.max(axis=1)

        for cell_type in unique_cell_types:
            pattern_max = pattern_max_series[cell_type]
            
            # GATE 2: Fold-Change Specificity
            other_cells_mean = pattern_max_series.drop(cell_type).median()
            if pattern_max < (other_cells_mean * FOLD_CHANGE_MIN):
                continue
                
            # GATE 3 & 4: Absolute Floor and Relative Dominance
            gene_total = total_gene_cpm_matrix.loc[cell_type].sum()
            if pattern_max >= ABSOLUTE_CPM_MIN and gene_total > 0:
                relative_share = pattern_max / gene_total
                if relative_share >= RELATIVE_THRESHOLD:
                    rbp_cell_genes[rbp][cell_type].add(gene)

# Format Output
records = []
for rbp, cell_types_dict in rbp_cell_genes.items():
    for cell_type, active_genes in cell_types_dict.items():
        records.append({
            'RBP': rbp,
            'Cell_Type': cell_type,
            'Active_Genes': ", ".join(sorted(list(active_genes))),
            'Target_Gene_Count': len(active_genes),
            'Driven_Destinations': rbp_destinations.get(rbp, "Unknown")
        })

df_out = pd.DataFrame(records)
if not df_out.empty:
    df_out = df_out.sort_values(by=['RBP', 'Target_Gene_Count'], ascending=[True, False]).reset_index(drop=True)

os.makedirs(os.path.dirname(output_csv), exist_ok=True)
df_out.to_csv(output_csv, index=False)

print(f"\n--- PIPELINE COMPLETE ---")
print(f"Filtered networks saved to: {output_csv}")

Loading data and building pseudo-bulk matrices...
Executing Pattern Clustering and Subtraction...
Applying Assignment Gates (Absolute, Relative, Housekeeping Silencer, Fold-Change)...

--- PIPELINE COMPLETE ---
Filtered networks saved to: ATTRACT/FIMO_Processed/RBP_Lists/RBP-CellType_lists.csv


In [31]:
input_csv = "ATTRACT/FIMO_Processed/RBP_Lists/RBP-CellType_lists.csv"
output_csv = "ATTRACT/FIMO_Processed/RBP_Lists/RBP-CellType_lists.csv"

print(f"Loading network data from {input_csv}...")
df = pd.read_csv(input_csv)

initial_rows = len(df)
df_cleaned = df[df['RBP'] != 'AGO2_C1'].copy()
final_rows = len(df_cleaned)

os.makedirs(os.path.dirname(output_csv), exist_ok=True)
df_cleaned.to_csv(output_csv, index=False)

print(f"Successfully dropped AGO2_C1.")
print(f"Network trimmed from {initial_rows} to {final_rows} specific interactions.")
print(f"Cleaned dataset saved to: {output_csv}")

Loading network data from ATTRACT/FIMO_Processed/RBP_Lists/RBP-CellType_lists.csv...
Successfully dropped AGO2_C1.
Network trimmed from 250 to 235 specific interactions.
Cleaned dataset saved to: ATTRACT/FIMO_Processed/RBP_Lists/RBP-CellType_lists.csv
